### Transformar Datos "Employees" - Expandir Arrays
1. Acceder a los elementos del objeto JSON
2. Eliminar elementos duplicados del array
3. Anular el anidamiento del array
4. Asignar al campo "status" los siguientes valores descriptivos:<br>
(0: Inactive, 1: Active)
5. Escribir los datos "transformados" en el schema "silver"

In [0]:
df_employees_json = spark.read.table("zenviro.silver.employees_json_py")
display(df_employees_json)

#### 1. Acceder a los elementos del objeto JSON
> `<column_name.object>`

In [0]:
df_employees_item = (
    df_employees_json
    .select(
        "json_value.employee_id",
        "json_value.department_id",
        "json_value.job_position_id",
        "json_value.name",
        "json_value.gender",
        "json_value.birth_date",
        "json_value.hire_date",
        "json_value.email",
        "json_value.phone",
        "json_value.status"
    )
)
display(df_employees_item)

#### 2. Eliminar elementos duplicados del array
[Doc - Función "array_distinct"](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_distinct.html)

In [0]:
from pyspark.sql.functions import array_distinct

df_employees_drop_duplicates = (
    df_employees_json
    .select(
        "json_value.employee_id",
        "json_value.department_id",
        "json_value.job_position_id",
        array_distinct("json_value.name").alias("name"),
        "json_value.gender",
        "json_value.birth_date",
        "json_value.hire_date",
        "json_value.email",
        "json_value.phone",
        "json_value.status"
    )
)
display(df_employees_drop_duplicates)

#### 3. Anular el anidamiento del array
[Doc - Función "explode"](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)

In [0]:
from pyspark.sql.functions import explode

df_employees_exploded = (
    df_employees_drop_duplicates
    .select(
        "employee_id",
        "department_id",
        "job_position_id",
        explode("name").alias("name"),
        "gender",
        "birth_date",
        "hire_date",
        "email",
        "phone",
        "status"
    )
)
display(df_employees_exploded)

In [0]:
from pyspark.sql.functions import col

df_employees_normalized = (
    df_employees_exploded
    .select(
        "employee_id",
        "department_id",
        "job_position_id",
        "name.first_name",
        "name.last_name",
        "gender",
        col("birth_date").cast("date").alias("birth_date"),
        col("hire_date").cast("date").alias("hire_date"),
        "email",
        "phone",
        "status"
    )
)
display(df_employees_normalized)

#### 4. Asignar al campo "status" los siguientes valores descriptivos:<br>
(0: Inactive, 1: Active)

In [0]:
from pyspark.sql.functions import when

df_employees_change_status = (
    df_employees_normalized
    .select(
        "employee_id",
        "department_id",
        "job_position_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "hire_date",
        "email",
        "phone",
        when(col("status") == 0, "Inactive")
        .when(col("status") == 1, "Active")
        .alias("status")
    )
)
display(df_employees_change_status)

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
df_employees_change_status.writeTo("zenviro.silver.employees_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.employees_py"))